In [ ]:
print("Adaptive Quiz Model Initialized Successfully")

Adaptive Quiz Model Initialized Successfully


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print("-", filename)

Saving difficulty.csv to difficulty.csv
Saving interaction_telemetry (6).csv to interaction_telemetry (6).csv
Saving concept_event_mapping (3).csv to concept_event_mapping (3).csv

Uploaded files:
- difficulty.csv
- interaction_telemetry (6).csv
- concept_event_mapping (3).csv


In [ ]:
import os
import glob

csv_files = glob.glob("*.csv")

print("CSV files found:")
for file in csv_files:
    print("-", file)


def find_file(keyword):
    for file in csv_files:
        if keyword.lower() in file.lower():
            return file
    return None


difficulty_file = find_file("difficulty")
telemetry_file = find_file("interaction")
mapping_file = find_file("concept")


# Load datasets
difficulty_df = pd.read_csv(difficulty_file)
telemetry_df = pd.read_csv(telemetry_file)
mapping_df = pd.read_csv(mapping_file)


print("\nDatasets loaded successfully!")

print("\nDifficulty Dataset:", difficulty_file)
print("Shape:", difficulty_df.shape)

print("\nInteraction Telemetry Dataset:", telemetry_file)
print("Shape:", telemetry_df.shape)

print("\nConcept Event Mapping Dataset:", mapping_file)
print("Shape:", mapping_df.shape)

CSV files found:
- difficulty.csv
- interaction_telemetry (6).csv
- concept_event_mapping (3).csv

Datasets loaded successfully!

Difficulty Dataset: difficulty.csv
Shape: (7, 13)

Interaction Telemetry Dataset: interaction_telemetry (6).csv
Shape: (123, 9)

Concept Event Mapping Dataset: concept_event_mapping (3).csv
Shape: (123, 9)


In [ ]:
print("========== DIFFICULTY DATASET ==========")
print(difficulty_df.columns.tolist())

print("\n========== INTERACTION TELEMETRY DATASET ==========")
print(telemetry_df.columns.tolist())

print("\n========== CONCEPT-EVENT MAPPING DATASET ==========")
print(mapping_df.columns.tolist())

========== DIFFICULTY DATASET ==========
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'Concept', 'ReplayCount', 'PauseCount', 'SeekBackwardCount', 'SeekForwardCount', 'UnderstandPauseCount', 'DifficultyScore', 'DifficultyLevel', 'NeedsRevision']

========== INTERACTION TELEMETRY DATASET ==========
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'EventType', 'EventTimestamp', 'EventDuration', 'PlaybackSpeed', 'PauseReason']

========== CONCEPT-EVENT MAPPING DATASET ==========
['StudentID', 'StudentName', 'CourseSubject', 'CourseName', 'EventType', 'EventTimestamp', 'Transcript', 'Concept', 'Prerequisite']


In [ ]:
print("DIFFICULTY DATASET")
display(difficulty_df.head())
print("INTERACTION TELEMETRY DATASET")
display(telemetry_df.head())
print("CONCEPT EVENT MAPPING DATASET")
display(mapping_df.head())

DIFFICULTY DATASET


,StudentID,StudentName,CourseSubject,CourseName,Concept,ReplayCount,PauseCount,SeekBackwardCount,SeekForwardCount,UnderstandPauseCount,DifficultyScore,DifficultyLevel,NeedsRevision
0,S24A003,Priya,ML,Intoduction to ML,Intoduction To Ml Fundamentals,0,0,13,4,0,28.0,High,True
1,S24A003,Priya,ML,Intoduction to ML,Search Fundamentals,0,2,2,1,0,6.5,Medium,False
2,S24A003,Priya,ML,Intoduction to ML,Introduction & Concepts,0,0,1,1,0,2.5,Easy,False
3,S24A003,Priya,ML,Well posed problem,Well Posed Problem Fundamentals,0,1,7,10,0,20.0,High,True
4,S24A003,Priya,ML,perspectives,Perspectives Fundamentals,0,0,5,7,0,13.5,High,True


INTERACTION TELEMETRY DATASET


,StudentID,StudentName,CourseSubject,CourseName,EventType,EventTimestamp,EventDuration,PlaybackSpeed,PauseReason
0,S24A003,Priya,ML,perspectives,SPEED_CHANGE,03:48,NaN,1.0,NaN
1,S24A003,Priya,ML,perspectives,SPEED_CHANGE,03:35,NaN,2.0,NaN
2,S24A003,Priya,ML,perspectives,PLAY,03:29,NaN,1.0,NaN
3,S24A003,Priya,ML,perspectives,PLAY,01:36,NaN,1.0,NaN
4,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,01:36,NaN,1.0,NaN


CONCEPT EVENT MAPPING DATASET


,StudentID,StudentName,CourseSubject,CourseName,EventType,EventTimestamp,Transcript,Concept,Prerequisite
0,S24A003,Priya,ML,Intoduction to ML,PLAY,00:00,Hello everyone welcome back to my youtube chan...,Intoduction To Ml Fundamentals,NaN
1,S24A003,Priya,ML,Intoduction to ML,PLAY,02:18,give any explicit instructions for the program...,Intoduction To Ml Fundamentals,Introduction & Concepts
2,S24A003,Priya,ML,Intoduction to ML,PAUSE,02:30,side let us see and on Amazon you search for s...,Search Fundamentals,Intoduction To Ml Fundamentals
3,S24A003,Priya,ML,Intoduction to ML,PLAY,02:30,side let us see and on Amazon you search for s...,Search Fundamentals,Intoduction To Ml Fundamentals
4,S24A003,Priya,ML,Intoduction to ML,PAUSE,02:40,dress or some shoes something you have searche...,Search Fundamentals,Intoduction To Ml Fundamentals


In [ ]:
datasets = {
    "Difficulty": difficulty_df,
    "Interaction Telemetry": telemetry_df,
    "Concept Event Mapping": mapping_df
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("\nShape:", df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:", df.duplicated().sum())


DIFFICULTY

Shape: (7, 13)

Data Types:
StudentID                object
StudentName              object
CourseSubject            object
CourseName               object
Concept                  object
ReplayCount               int64
PauseCount                int64
SeekBackwardCount         int64
SeekForwardCount          int64
UnderstandPauseCount      int64
DifficultyScore         float64
DifficultyLevel          object
NeedsRevision              bool
dtype: object

Missing Values:
StudentID               0
StudentName             0
CourseSubject           0
CourseName              0
Concept                 0
ReplayCount             0
PauseCount              0
SeekBackwardCount       0
SeekForwardCount        0
UnderstandPauseCount    0
DifficultyScore         0
DifficultyLevel         0
NeedsRevision           0
dtype: int64

Duplicate Rows: 0

INTERACTION TELEMETRY

Shape: (123, 9)

Data Types:
StudentID          object
StudentName        object
CourseSubject      object
CourseName 

In [ ]:
print("========== DIFFICULTY LEVELS ==========")
print(difficulty_df["DifficultyLevel"].value_counts())

print("\n========== NEEDS REVISION ==========")
print(difficulty_df["NeedsRevision"].value_counts())

print("\n========== TELEMETRY EVENT TYPES ==========")
print(telemetry_df["EventType"].value_counts())

print("\n========== PAUSE REASONS ==========")
print(telemetry_df["PauseReason"].value_counts(dropna=False))

print("\n========== MAPPING EVENT TYPES ==========")
print(mapping_df["EventType"].value_counts())

print("\n========== UNIQUE CONCEPTS ==========")
print(mapping_df["Concept"].unique())

========== DIFFICULTY LEVELS ==========
DifficultyLevel
High      3
Medium    2
Easy      2
Name: count, dtype: int64

========== NEEDS REVISION ==========
NeedsRevision
False    4
True     3
Name: count, dtype: int64

========== TELEMETRY EVENT TYPES ==========
EventType
PLAY              43
SEEK_FORWARD      36
SEEK_BACKWARD     35
SPEED_CHANGE       5
PAUSE              3
VIDEO_COMPLETE     1
Name: count, dtype: int64

========== PAUSE REASONS ==========
PauseReason
NaN                              120
Paused by student                  2
Interrupted by something else      1
Name: count, dtype: int64

========== MAPPING EVENT TYPES ==========
EventType
PLAY              43
SEEK_FORWARD      36
SEEK_BACKWARD     35
SPEED_CHANGE       5
PAUSE              3
VIDEO_COMPLETE     1
Name: count, dtype: int64

========== UNIQUE CONCEPTS ==========
['Intoduction To Ml Fundamentals' 'Search Fundamentals'
 'Introduction & Concepts' 'Well Posed Problem Fundamentals'
 'Perspectives Fundamentals'

In [ ]:
join_keys = [
    "StudentID",
    "CourseSubject",
    "CourseName",
    "EventType",
    "EventTimestamp"
]

merged_events = telemetry_df.merge(
    mapping_df[
        join_keys + ["Concept", "Transcript", "Prerequisite"]
    ],
    on=join_keys,
    how="left"
)

print("Telemetry rows:", len(telemetry_df))
print("Merged rows:", len(merged_events))

print("\nRows with concept mapping:")
print(merged_events["Concept"].notna().sum())

print("\nRows without concept mapping:")
print(merged_events["Concept"].isna().sum())

print("\nMerged preview:")
display(merged_events.head(10))

Telemetry rows: 123
Merged rows: 199

Rows with concept mapping:
199

Rows without concept mapping:
0

Merged preview:


,StudentID,StudentName,CourseSubject,CourseName,EventType,EventTimestamp,EventDuration,PlaybackSpeed,PauseReason,Concept,Transcript,Prerequisite
0,S24A003,Priya,ML,perspectives,SPEED_CHANGE,03:48,NaN,1.0,NaN,Perspectives Fundamentals,We have seen handwriting recognition problem.,Algorithm Definition & Properties
1,S24A003,Priya,ML,perspectives,SPEED_CHANGE,03:35,NaN,2.0,NaN,Algorithm Definition & Properties,Among all the algorithms that we have which al...,Perspectives Fundamentals
2,S24A003,Priya,ML,perspectives,PLAY,03:29,NaN,1.0,NaN,Algorithm Definition & Properties,So like that you have so many algorithms in ma...,Perspectives Fundamentals
3,S24A003,Priya,ML,perspectives,PLAY,01:36,NaN,1.0,NaN,Search Fundamentals,Now in the possible space of hypothesis we are...,Perspectives Fundamentals
4,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,01:36,NaN,1.0,NaN,Search Fundamentals,Now in the possible space of hypothesis we are...,Perspectives Fundamentals
5,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,01:46,NaN,1.0,NaN,Perspectives Fundamentals,the observed data of the data in the sense.,Search Fundamentals
6,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,01:56,NaN,1.0,NaN,Algorithm Definition & Properties,So this is the hypothesis in which we are havi...,Perspectives Fundamentals
7,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,02:06,NaN,1.0,NaN,Perspectives Fundamentals,You will select one according to your requirem...,Algorithm Definition & Properties
8,S24A003,Priya,ML,perspectives,PLAY,02:16,NaN,1.0,NaN,Perspectives Fundamentals,Whether the data is pure in the resource or yo...,Algorithm Definition & Properties
9,S24A003,Priya,ML,perspectives,SEEK_BACKWARD,02:16,NaN,1.0,NaN,Perspectives Fundamentals,Whether the data is pure in the resource or yo...,Algorithm Definition & Properties


In [ ]:
concept_event_counts = (
    merged_events
    .groupby(
        [
            "StudentID",
            "CourseSubject",
            "CourseName",
            "Concept"
        ]
    )
    .size()
    .reset_index(name="EventCount")
)

display(concept_event_counts)

,StudentID,CourseSubject,CourseName,Concept,EventCount
0,S24A003,ML,Intoduction to ML,Intoduction To Ml Fundamentals,29
1,S24A003,ML,Intoduction to ML,Introduction & Concepts,4
2,S24A003,ML,Intoduction to ML,Search Fundamentals,9
3,S24A003,ML,Well posed problem,Well Posed Problem Fundamentals,124
4,S24A003,ML,perspectives,Algorithm Definition & Properties,10
5,S24A003,ML,perspectives,Perspectives Fundamentals,21
6,S24A003,ML,perspectives,Search Fundamentals,2
